# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadsammad42/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

---



*One row = one what, over which dates? State it, then verify it below.*

Ans: Unit of analysis: One row represents the daily performance of one content item for one client on one report date.

Time window: I will use March 2026 (2026-03-01 through 2026-03-31) for this data contract and feature development. I use a mid-panel month so that the final month, June 2026, remains a sealed test/outcome window.

In [20]:
!pip -q install duckdb

In [21]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN not found. Make sure you added a Colab Secret "
        "named 'HF_TOKEN' and enabled Notebook access."
    )

# Create DuckDB connection
con = duckdb.connect()

# Authenticate DuckDB with Hugging Face
con.execute(
    f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
    """
)

print("DuckDB connection established successfully.")

DuckDB connection established successfully.


In [22]:
# Verify the claimed grain for March 2026.
# Expected grain:
# one row per client × content × report date.

con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COUNT(*) AS row_count
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/'
        'fact_content_daily_performance/month=2026-03/*.parquet'
    )
    GROUP BY
        client_hash_id,
        content_hash_id,
        report_date
    HAVING COUNT(*) > 1
    ORDER BY row_count DESC
    LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Ans: Feature: gsc_clicks, gsc_impressions, gsc_ctr, gsc_avg_position, and content age. These describe the page's search performance and freshness and are available at the decision moment.

Label / proxy: is_declining_label. This is used as a proxy for content refresh opportunity because declining pages are candidates for review. It is the outcome/proxy, not a feature.

Context: client_hash_id, content_hash_id, and report_date. These identify the client, content item, and observation date. They are used for grouping, joining, and time-based analysis, not as model features.

Excluded: trend_direction and trend_pct because is_declining_label is derived from trend_direction, so using these fields as features would leak information from the label into the inputs. I also exclude ga4_data_available because it indicates data availability rather than content performance.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Ans: Verification: I verify the data contract using three queries on the March 2026 partition. The first checks that the claimed grain has no duplicate client-content-date combinations. The second checks the number of rows and the date range. The third checks GSC data availability using

In [24]:
# Query 1: Verify the grain.
# Expected grain: one row per client × content × report date.

con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COUNT(*) AS row_count
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/'
        'fact_content_daily_performance/month=2026-03/*.parquet'
    )
    GROUP BY
        client_hash_id,
        content_hash_id,
        report_date
    HAVING COUNT(*) > 1
    ORDER BY row_count DESC
    LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count


In [25]:
# Query 2: Verify the row count and date span for March 2026.

con.sql("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/'
        'fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [26]:
# Query 3: Verify GSC data availability.
# The assignment specifically requires IS TRUE.

con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS NOT TRUE
        ) AS gsc_unavailable_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/'
        'fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,gsc_unavailable_rows
0,9841378,3611061,6230317


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Ans: Data limitation: This data can show page performance and search/traffic signals, but it cannot tell us whether a content team actually refreshed a page or whether a refresh caused future improvement. The history is also unbalanced across clients because different clients may have different data start dates. In addition, some early rows may have GSC data without GA4 data, so comparisons across all pages are not always directly equivalent. Finally, using a fixed monthly window can overlap with performance trends from nearby periods.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.